In [0]:
import requests
import logging
from datetime import datetime
from pyspark.sql import SparkSession
import pandas as pd

In [0]:
# Inicializa Spark
spark = SparkSession.builder.getOrCreate()

In [0]:
# API + Config
API_URL = "https://api.openbrewerydb.org/v1/breweries"
SCHEMA = "bronze"
TABLE_NAME = "raw_breweries"

In [0]:
def fetch_breweries():
    all_breweries = []
    page = 1
    per_page = 50

    while True:
        response = requests.get(API_URL, params={"page": page, "per_page": per_page})
        if response.status_code != 200:
            raise Exception(f"Erro na API: {response.status_code}")
        
        data = response.json()
        if not data:
            break

        all_breweries.extend(data)
        page += 1

    return all_breweries

In [0]:
def run():
    breweries = fetch_breweries()

    # Converte para Pandas, depois Spark
    pdf = pd.DataFrame(breweries)   # pdf, pandas dataframe
    df = spark.createDataFrame(pdf) # df, dataframe do spark

    # Cria a tabela no metastore
    df.write.mode("overwrite").format("delta").saveAsTable(f"{SCHEMA}.{TABLE_NAME}")

In [0]:
if __name__ == "__main__":
    run()